# 15 Spark Internals: Interview Q&A

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Consolidate the whole <code>02_spark_internals</code> folder into <b>spoken answers</b> you can give in an interview: the execution flow you should be able to draw, six 2-minute explanations, rapid-fire questions, and scenario questions. Then practise them out loud.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Knowing internals and <b>explaining</b> them under pressure are different skills. Interviewers listen for structure (what happens, why, trade-offs, an example) and for correct vocabulary (driver, stage, shuffle, partition, exchange, AQE). This notebook is your rehearsal script.
</div>

## How to use this notebook

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b>
<ol><li>Read a question, cover the answer, and say yours <b>out loud</b> (time yourself)</li><li>Compare with the model answer: did you cover what happens, why, and one trade-off or example?</li><li>Run the quiz cell at the end to get random questions</li><li>Log weak answers in <code>interview_prep/mock_interview_log.md</code> and revisit the lesson</li></ol>
</div>

## The execution flow you must be able to draw

```
 Application (SparkSession in the DRIVER)
        │  your code builds a logical plan (lazy)
        ▼
 ACTION (count / write / collect)  ──▶  JOB
        │  Catalyst: analyze → optimize → physical plan  (AQE re-plans at runtime)
        ▼
 DAG scheduler: split the plan at shuffles (Exchange)  ──▶  STAGES
        │
        ▼
 Task scheduler: one TASK per partition, sent to EXECUTOR cores
        │       map tasks write shuffle files ──▶ reduce tasks fetch them
        ▼
 Results to the driver, or written to storage  ──▶  OUTPUT
```

Practise drawing this on paper in under a minute, naming each box.

## The six 2-minute explanations

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What happens when I run <code>df.groupBy("country").count()</code> and then call <code>.show()</code>?</b><br>
Nothing runs on <code>groupBy</code> or <code>count</code>: they only build a logical plan. <code>show()</code> is the action, so the driver creates a job. Catalyst optimizes the plan, for example pruning unused columns and pushing filters into the scan, and plans a two-step aggregation. The DAG scheduler cuts the plan at the shuffle into two stages. In stage one, each task reads one input partition and does a partial count per country inside that partition, then writes the partial results to shuffle files, hashed by country. In stage two, each task fetches the blocks for its countries from every map task and adds up the partial counts. AQE may coalesce the 200 shuffle partitions into a few because the result is small. Finally <code>show()</code> fetches the first 20 rows back to the driver.<br><br>

<b>🎤 2. How does a shuffle work, and why is it expensive?</b><br>
A shuffle redistributes rows so that all rows with the same key land in the same partition. Each map task computes a target partition for every row, buffers and sorts by target, and writes one data file plus an index to local disk. Once all map tasks are done, each reduce task fetches its blocks from every map output over the network and merges them. It's expensive because of serialization, disk writes and reads, network transfer, memory pressure that can cause spill, and the barrier between stages. I reduce it by filtering and selecting before wide operations, broadcasting small join sides, removing redundant repartitions and sorts, and handling skew.<br><br>

<b>🎤 3. Broadcast join vs sort-merge join?</b><br>
A broadcast hash join collects the small table to the driver and sends a copy to every executor, which builds a hash table. The large table is then joined partition by partition without being shuffled, which makes it the fastest strategy when one side is small. Spark uses it below the 10 MB threshold, with a hint, or when AQE sees a small side at runtime. A sort-merge join shuffles both sides by the join key, sorts each partition, and merges them. It scales to any size and can spill to disk, but costs two shuffles and two sorts. It's the default for large equi-joins. Broadcasting a table that's too big can crash the driver, and you can't broadcast the preserved side of an outer join.<br><br>

<b>🎤 4. What is lazy evaluation and why does Spark use it?</b><br>
Transformations only record steps in a plan. Execution starts when an action is called. Because Spark sees the whole query first, Catalyst can push filters into the scan, prune unused columns, combine and reorder steps, choose join strategies and fuse narrow operators into one generated function, and <code>limit</code> can stop early. The costs: each action recomputes from the source unless results are materialized, and data errors only appear at the action.<br><br>

<b>🎤 5. Explain jobs, stages and tasks.</b><br>
Each action creates a job. The job is split into stages at shuffle boundaries, which are the <code>Exchange</code> operators in the plan, so a simple query with one groupBy has two stages. Each stage runs as a set of tasks, one per partition, and each task runs on one executor core. If there are more tasks than cores, they run in waves, and the stage finishes when its slowest task does, which is why skew matters.<br><br>

<b>🎤 6. What is Adaptive Query Execution?</b><br>
AQE re-optimizes the plan at runtime. After each shuffle stage finishes, Spark knows the real partition sizes and re-plans the rest: it coalesces small shuffle partitions into fewer tasks, converts a sort-merge join into a broadcast join when one side turns out small, and splits skewed partitions in joins. It's on by default in Spark 3 and always on for Databricks serverless. <code>explain()</code> shows the initial plan, and the Spark UI or query profile shows the final one.
</div>

## Rapid-fire questions (one or two sentences each)

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What does the driver do?</b><br>
It runs the main program, holds the SparkSession, plans the query, schedules tasks on executors, and collects results.<br><br>

<b>🎤 2. What is an executor?</b><br>
A process on a worker node that runs tasks on its cores and stores shuffle and cached data for one application.<br><br>

<b>🎤 3. Narrow vs wide transformation?</b><br>
Narrow: each output partition depends on one input partition, so there's no shuffle. Wide: it needs data from many partitions, so there's a shuffle and a new stage.<br><br>

<b>🎤 4. <code>repartition</code> vs <code>coalesce</code>?</b><br>
<code>repartition</code> shuffles to create balanced partitions and can increase or decrease the count. <code>coalesce</code> merges partitions without a shuffle and can only decrease the count.<br><br>

<b>🎤 5. What is <code>spark.sql.shuffle.partitions</code>?</b><br>
The number of partitions after a shuffle, 200 by default. AQE coalesces small ones at runtime.<br><br>

<b>🎤 6. <code>cache</code> vs <code>persist</code>?</b><br>
<code>cache</code> is <code>persist</code> with <code>MEMORY_AND_DISK</code>. Both are lazy and filled by the next action. They aren't supported on serverless, where you write a Delta table instead.<br><br>

<b>🎤 7. What is Catalyst?</b><br>
Spark SQL's optimizer: analysis, rule-based logical optimization, physical planning with statistics, then code generation.<br><br>

<b>🎤 8. What is Tungsten?</b><br>
The execution engine work on CPU and memory efficiency: binary row format, whole-stage code generation and vectorized reads.<br><br>

<b>🎤 9. Why are DataFrames faster than RDDs?</b><br>
The schema lets Catalyst optimize the whole query and Tungsten use compact binary rows and generated code. RDD lambdas are opaque and run as written.<br><br>

<b>🎤 10. Why avoid Python UDFs?</b><br>
Rows are serialized to Python workers and back, and the optimizer can't push down, fold or code-generate them.<br><br>

<b>🎤 11. What is data skew?</b><br>
An uneven distribution of rows across partitions, usually because of hot keys, which makes a few tasks much slower than the rest.<br><br>

<b>🎤 12. What is spill?</b><br>
Data written to disk because it didn't fit in execution memory during a sort, aggregation or join.<br><br>

<b>🎤 13. What does <code>Exchange</code> mean in a plan?</b><br>
A shuffle, and therefore a stage boundary.<br><br>

<b>🎤 14. What is whole-stage code generation?</b><br>
Fusing a chain of operators into one generated Java function, shown as <code>*(n)</code> in plans.<br><br>

<b>🎤 15. What is Spark Connect?</b><br>
A client-server protocol where a thin client sends plans to a remote Spark driver. It's used by Databricks serverless notebooks, so there's no <code>sparkContext</code>.<br><br>

<b>🎤 16. What is Photon?</b><br>
Databricks' native vectorized C++ query engine, which accelerates SQL and DataFrame operations without code changes.
</div>

## Scenario questions

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. One task in a stage takes 30 minutes while the others take 30 seconds. What do you do?</b><br>
That's skew. I confirm it in the stage's task summary or the query profile, then find the hot key with a <code>groupBy(key).count()</code>. Fixes: let AQE skew join handling split it, broadcast the smaller side if possible, filter out or separately process the hot key, or salt the key by adding a random suffix and aggregating in two steps.<br><br>

<b>🎤 2. A job writes 20,000 tiny files. Why, and how do you fix it?</b><br>
Each task writes a file per output partition it touches, so many partitions, or high-cardinality <code>partitionBy</code> columns, produce tiny files. I coalesce or repartition to a sensible number before writing, repartition by the partition column, choose coarser partition columns, and on Delta run <code>OPTIMIZE</code> or enable auto compaction, or use liquid clustering.<br><br>

<b>🎤 3. A join is slow. How do you debug it?</b><br>
I check the plan for the join strategy and whether the small side was broadcast, the size of the shuffle on each side, skew in the join key, rows exploding after the join because of duplicate keys, and whether filters and column pruning happen before the join. Then I fix the biggest issue: broadcast, filter earlier, deduplicate keys, or handle skew.<br><br>

<b>🎤 4. The driver crashes with out-of-memory. What are the likely causes?</b><br>
<code>collect()</code> or <code>toPandas()</code> on large data, a very large broadcast, too many tiny tasks or files for the driver to track, or a huge <code>display()</code>. I aggregate or limit before bringing data back, and avoid oversized broadcasts.<br><br>

<b>🎤 5. The same notebook gives different results when re-run. Why?</b><br>
Non-deterministic expressions such as <code>uuid()</code> are recomputed per action, unordered operations such as <code>dropDuplicates</code> or <code>first</code> pick arbitrary rows, or sorts have ties. I add deterministic ordering and tie-breakers, and materialize results that must be stable.
</div>

## Quiz: random questions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Picks 5 random questions from the lists above. Answer each out loud before revealing the next.<br><br>
<b>Why it matters</b><br>Random order is closer to a real interview than reading the list top to bottom.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
5 numbered questions. Run the cell again for a new set.
</div>

In [0]:
import random

questions = [
    "What happens when I run df.groupBy('country').count().show()?",
    "How does a shuffle work, and why is it expensive?",
    "Broadcast join vs sort-merge join?",
    "What is lazy evaluation and why does Spark use it?",
    "Explain jobs, stages and tasks.",
    "What is Adaptive Query Execution?",
    "repartition vs coalesce?",
    "cache vs persist? What do you do on serverless?",
    "What is Catalyst? What is Tungsten?",
    "Why are DataFrames faster than RDDs?",
    "Why avoid Python UDFs?",
    "One task takes 30 minutes while the others take 30 seconds. What do you do?",
    "A job writes 20,000 tiny files. Why, and how do you fix it?",
    "A join is slow. How do you debug it?",
    "The driver runs out of memory. What are the likely causes?",
    "What does Exchange mean in a physical plan?",
    "What is Spark Connect, and what changes on serverless?",
]
for i, q in enumerate(random.sample(questions, 5), 1):
    print(f"{i}. {q}")

## Prove it: one query, every concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs one query and prints its plan, so you can narrate it as you would in an interview: the job, the stages, the shuffle, the join strategy and AQE.<br><br>
<b>Why it matters</b><br>Pointing at a real plan while explaining is the most convincing way to show you understand internals.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A plan with <code>AdaptiveSparkPlan</code>, a <code>BroadcastHashJoin</code> (small dimension), and <code>HashAggregate</code> → <code>Exchange</code> → <code>HashAggregate</code>. Try narrating it bottom-up.
</div>

In [0]:
from pyspark.sql import functions as F

sales = spark.range(0, 2_000_000, 1, numPartitions=8).select(
    (F.col("id") % 1000).alias("store_id"), (F.col("id") % 997).alias("amount"))
stores = spark.range(1000).select(F.col("id").alias("store_id"), (F.col("id") % 5).alias("region"))

report = sales.filter("amount > 100").join(stores, "store_id").groupBy("region").agg(F.sum("amount").alias("revenue"))
report.explain()
report.orderBy("region").show()

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which statement about Spark execution is correct?</b><br>
A. Every transformation immediately launches a job<br>
B. An action triggers a job, which is divided into stages at shuffle boundaries, and each stage runs one task per partition<br>
C. Each stage always has exactly 200 tasks<br>
D. Tasks run on the driver<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A query on serverless compute is slow, and the query profile shows a sort-merge join where one input has only 300 rows after filtering. What explains this, and what happens at runtime?</b><br>
A. Sort-merge joins are always used on serverless<br>
B. The static plan estimated a large size. AQE can convert the join to a broadcast join at runtime once it sees the real size, and a broadcast hint forces it upfront<br>
C. The table must be cached first<br>
D. The shuffle partition count is too low<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. Which change usually reduces shuffle volume the most?</b><br>
A. Increasing the number of executors<br>
B. Selecting only needed columns and filtering rows before wide transformations<br>
C. Calling <code>cache()</code> before <code>groupBy</code><br>
D. Using <code>orderBy</code> before <code>groupBy</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Draw the execution flow from memory and compare it with the diagram at the top</li><li>Record yourself answering the six 2-minute explanations. Each should take 60–120 seconds</li><li>Run the quiz cell three times and answer every question out loud</li><li>Narrate the plan of <code>report</code> bottom-up: scan, filter, broadcast join, partial aggregate, exchange, final aggregate</li><li>Add your two weakest answers to <code>interview_prep/mock_interview_log.md</code> with notes on what to improve</li></ol>
</div>

In [0]:
# Use this cell for notes, or to run your own experiments while practising.

## Summary

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Draw it: driver → action → job → stages (at shuffles) → tasks (per partition) → executors → output</li><li>Six core explanations: groupBy flow, shuffle, broadcast vs sort-merge, lazy evaluation, jobs/stages/tasks, AQE</li><li>Structure every answer: what happens, why, a trade-off, an example</li><li>Use the right words: exchange, partition, stage, task, spill, skew, broadcast, AQE, Catalyst, Tungsten, Photon</li><li>Back claims with plans and profiles: <code>explain()</code>, the query profile, the Spark UI</li></ul>
</div>

| Lesson | Topic |
|---|---|
| 01 | Driver and executors |
| 02 | Jobs, stages, tasks |
| 03 | Transformations and actions |
| 04 | Lazy evaluation |
| 05 | DAG and query plans |
| 06 | Narrow and wide |
| 07 | Shuffle |
| 08 | Partitioning and parallelism |
| 09 | Join strategies |
| 10 | Caching |
| 11 | Serialization |
| 12 | Catalyst and Tungsten |
| 13 | AQE |
| 14 | Spark UI and query profile |

## Git commit

```
git add 02_spark_internals/15_internals_interview_qa.ipynb
git commit -m "add 02_15 internals interview qa notebook"
```